In [ ]:
import sys
sys.path.append("..")  # pour importer le dossier sources/, situé à la racine du dépôt

In [ ]:
# Chargement des 8 sessions : téléchargées une seule fois, puis relues depuis data/raw/
from sources.chargement_data import charger_parcoursup, charger_poursuite, charger_ips, DATASETS
from sources.cleaning_data import preparer_base, harmoniser_ips, preparer_poursuite
from sources.jointures import joindre_ips, panel_academies
from sources.indicateurs import taux_poursuite, part_poursuite_autre_academie_idf, part_admis_meme_academie

dfs = {annee: charger_parcoursup(annee) for annee in DATASETS}

Préparation de la base : identifiant de formation suivi dans le temps (`code`), sélectivité (`selective`) et filière (`filiere`) harmonisées, puis empilement des sessions. Les choix sont justifiés dans `notebooks/exploration.ipynb`.

In [ ]:
base = preparer_base(dfs)

print(base.shape)
print("sélectivité manquante :", base["selective"].isna().sum())
base.groupby("session").size()

## Enrichissement 1 : l'indice de position sociale (IPS) des lycées

L'IPS, publié par l'Éducation nationale, résume le milieu social des élèves d'un lycée : plus il est élevé, plus le milieu est favorisé. Beaucoup de CPGE et de BTS sont installés dans des lycées. On peut donc rattacher à chaque formation l'IPS du lycée qui l'accueille, grâce au code UAI et à l'année. Les trois fichiers publiés (2016-2021, 2022, 2023-2025) n'ont pas les mêmes formats : on les harmonise avant la jointure.

In [ ]:
ips = harmoniser_ips(charger_ips())
base = joindre_ips(base, ips)

# Part des formations qui trouvent l'IPS de leur lycée, par filière (en %)
avec_ips = base["ips_lycee"].notna()
(avec_ips.groupby(base["filiere"]).mean() * 100).round(1).sort_values(ascending=False)

94 % des CPGE et 86 % des BTS trouvent l'IPS de leur lycée. Les universités, les IUT et les écoles ne sont pas des lycées : la jointure ne les concerne pas. L'analyse fondée sur l'IPS portera donc sur les CPGE et les BTS.

## Enrichissement 2 : le taux de poursuite des néo-bacheliers dans le supérieur (2010-2025)

Ce fichier du ministère donne, pour chaque académie où le bac a été obtenu, chaque année et chaque voie du bac, le nombre de bacheliers et le nombre de ceux qui poursuivent dans le supérieur. Il commence en 2010 : on peut comparer la période APB (jusqu'en 2017) à la période Parcoursup (depuis 2018).

In [ ]:
poursuite = preparer_poursuite(charger_poursuite())

# Taux de poursuite (en %) par année et par voie du bac : une colonne par voie
taux_poursuite(poursuite, ["annee", "voie"]).unstack()

Le taux de poursuite des bacheliers professionnels passe de 34,8 % en 2010 à 38,6 % en 2017, puis à 46,5 % en 2025. Celui des bacheliers généraux (environ 93 %) et technologiques (environ 80 %) reste stable.

On relie ensuite ce fichier aux admis de Parcoursup, par académie et par année. On obtient un tableau avec une ligne par académie et par année : c'est la jointure entre les deux sources. Attention : Parcoursup donne l'académie de la formation, et le fichier de poursuite l'académie du bac.

In [ ]:
panel = panel_academies(base, poursuite)

print(panel.shape, "-", panel["academie"].nunique(), "académies")
panel.head()

## Zoom Île-de-France : le secteur unique de 2019

Depuis la session 2019, les académies de Paris, Créteil et Versailles forment un seul secteur dans Parcoursup. On regarde l'effet de ce changement dans les deux sources :
- Parcoursup : la part des admis venant de l'académie de Paris, dans les formations parisiennes ;
- poursuite d'études : la part des bacheliers franciliens qui poursuivent dans une autre académie d'Île-de-France.

In [ ]:
import pandas as pd

# Les deux indicateurs côte à côte, en %, de 2016 à 2025 (Parcoursup commence en 2018)
idf = pd.DataFrame({
    "admis de l'académie de Paris dans les formations parisiennes": part_admis_meme_academie(base, "Paris"),
    "bacheliers franciliens poursuivant dans une autre académie d'IDF": part_poursuite_autre_academie_idf(poursuite),
})
idf.loc[2016:]